# Kerchunk vs netCDF Read Performance for 3hr Datasets with Slower Kerchunk Reads

Kerchunk read performance for 3hr datasets can be slower than native netCDF when **reference counts are very large**, causing `open_dataset` to become dominated by Python-level metadata processing rather than I/O.

## Overview

- Compares read performance of Kerchunk references and native netCDF files for 3hr datasets.
- Focuses on cases where Kerchunk read times are slower than `open_mfdataset`, despite identical logical dimension order.
- Identifies **large reference counts** as the primary cause of slow Kerchunk opens.

## Key Findings

### 1. High Reference Counts Are the Root Cause

- Kerchunk reference files contain **millions of references** (often 7–10M+).
- Each reference corresponds to a single underlying **HDF5 chunk**.
- Reference counts grow as the product of chunk counts across dimensions:
  - `references ≈ time_chunks × lat_chunks × lon_chunks × variables`
- Long time series combined with fine spatial chunking cause **reference explosion**.
- `open_dataset` must parse and materialize all references in Python, resulting in a large up-front cost.
- At this scale, Kerchunk performance becomes **metadata-bound**, not I/O-bound.

### 2. Dimension Ordering

- The `pr` variable has the **same dimension order** (`time, lat, lon`) in both Kerchunk references and raw netCDF files.
- Dimension order does **not** explain the observed performance gap.

### 3. Chunking

- Kerchunk exposes the **original HDF5 chunking** rather than defining logical (Zarr-style) chunks.
- Fine-grained chunking combined with long time series leads directly to **very large reference counts**.
- Logical compute chunking is applied later (e.g., via Dask rechunking), not at open time.

## Conclusion

- For these 3hr datasets, Kerchunk is slower than native netCDF primarily because:
- Millions of references must be parsed at open time
- Metadata reconstruction happens in Python
- Chunk semantics amplify open-time bookkeeping costs
- When netCDF files are already well-chunked and stored locally, `open_mfdataset` is often the more efficient choice unless reference counts can be substantially reduced.

## Overall Takeaway

Kerchunk performance is dominated by **reference count**, not dimension order, for long, finely chunked 3hr datasets. Each HDF5 chunk becomes one reference, so reference counts scale multiplicatively with chunking and time length. When reference counts reach the millions, `open_dataset` becomes metadata-bound and slower than `open_mfdataset`. Kerchunk is best suited for reducing I/O costs in remote or cloud-native workflows; for local, well-chunked netCDF files with long time series, native netCDF access is often the better-performing option.


In [2]:
import json

import pandas as pd
import xarray as xr
from IPython.display import HTML

# Prevent truncation of strings
pd.set_option("display.max_colwidth", None)
# Display floats with two decimal places
pd.options.display.float_format = "{:.2f}".format

## Load Raw Metrics


In [3]:
df_raw = pd.read_csv(
    "riotai/results/20260126_130127/kerchunk_vs_netcdf_raw_20260126_130127.csv"
)

# Add a new column for the difference between kerchunk_time and netcdf_time
df_raw["time_difference"] = df_raw["kerchunk_time"] - df_raw["netcdf_time"]

## Check the longest Kerchunk runtimes (Outliers)


In [16]:
df_raw_sorted = df_raw.sort_values(by="time_difference", ascending=False)

df_3hr_slow = (
    df_raw[df_raw["frequency"] == "3hr"]
    .sort_values(by="time_difference", ascending=False)
    .head(n=8)
)

keys = df_3hr_slow["json"].tolist()

HTML(
    f"""
<div style="height:225px; overflow-y:scroll;">
    {df_3hr_slow.to_html(max_rows=None, max_cols=None, notebook=True)}
</div>
"""
)

,frequency,json,num_netcdf_files,timesteps,dims,kerchunk_time,netcdf_time,time_difference
101,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/control-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.control-1950.r1i1p2f1.3hr.pr.gr.v20181119.kerchunk.json,100,292200,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 292200}",69.24,49.71,19.53
127,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/control-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.control-1950.r2i1p2f1.3hr.pr.gr.v20190722.kerchunk.json,101,295120,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 295120}",69.84,50.54,19.30
105,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r2i1p2f1.3hr.pr.gr.v20190625.kerchunk.json,65,189928,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189928}",45.85,27.76,18.09
100,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-present/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-present.r3i1p1f1.3hr.pr.gr.v20190509.kerchunk.json,65,189928,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189928}",43.99,27.50,16.49
119,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r1i1p2f1.3hr.pr.gr.v20181212.kerchunk.json,65,189927,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189927}",42.91,30.00,12.91
93,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-future/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-future.r3i1p1f1.3hr.pr.gr.v20190713.kerchunk.json,36,105192,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 105192}",23.94,17.20,6.74
104,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-future/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-future.r1i1p1f1.3hr.pr.gr.v20190514.kerchunk.json,35,102272,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 102272}",24.31,19.41,4.90
120,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-present/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P.highresSST-present.r2i1p1f1.3hr.pr.gr.v20190930.kerchunk.json,65,189928,"{'lat': 256, 'bnds': 2, 'lon': 512, 'time': 189928}",16.16,15.44,0.72


### Get the number of reference files for each slow 3-hr JSON.

In [5]:
def get_reference_count(json_path):
    with open(json_path, "r") as f:
        ref = json.load(f)

        return len(ref["refs"])


df_3hr_slow["reference_count"] = df_3hr_slow["json"].apply(get_reference_count)

In [15]:
HTML(
    f"""
<div style="height:225px; overflow-y:scroll;">
    {df_3hr_slow.to_html(max_rows=None, max_cols=None, notebook=True)}
</div>
"""
)

,frequency,json,num_netcdf_files,timesteps,dims,kerchunk_time,netcdf_time,time_difference,reference_count
101,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/control-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.control-1950.r1i1p2f1.3hr.pr.gr.v20181119.kerchunk.json,100,292200,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 292200}",69.24,49.71,19.53,10811536
127,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/control-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.control-1950.r2i1p2f1.3hr.pr.gr.v20190722.kerchunk.json,101,295120,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 295120}",69.84,50.54,19.30,10919432
105,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r2i1p2f1.3hr.pr.gr.v20190625.kerchunk.json,65,189928,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189928}",45.85,27.76,18.09,7027324
100,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-present/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-present.r3i1p1f1.3hr.pr.gr.v20190509.kerchunk.json,65,189928,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189928}",43.99,27.50,16.49,7027324
119,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r1i1p2f1.3hr.pr.gr.v20181212.kerchunk.json,65,189927,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189927}",42.91,30.00,12.91,7027323
93,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-future/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-future.r3i1p1f1.3hr.pr.gr.v20190713.kerchunk.json,36,105192,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 105192}",23.94,17.20,6.74,3892128
104,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-future/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-future.r1i1p1f1.3hr.pr.gr.v20190514.kerchunk.json,35,102272,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 102272}",24.31,19.41,4.90,3784232
120,3hr,/global/cfs/projectdirs/m4931/kerchunk/pr/highresSST-present/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P.highresSST-present.r2i1p1f1.3hr.pr.gr.v20190930.kerchunk.json,65,189928,"{'lat': 256, 'bnds': 2, 'lon': 512, 'time': 189928}",16.16,15.44,0.72,1879276


**Key observations:**

- As **reference_count increases**, the **time difference (kerchunk − netCDF)** consistently increases.
- Datasets with ~3.8M references show a ~5–7 s gap.
- Datasets with ~7.0M references show a ~13–18 s gap.
- Datasets with ~10.8–10.9M references show the largest gap (~19–20 s).
- The growth in time difference closely tracks the growth in reference_count.

**Takeaway:** Kerchunk open time scales with the number of references; higher reference counts lead to a proportionally larger performance penalty relative to netCDF.


## Analyze the top three slowest file for dimensions and chunking strategy


In [4]:
# Load the JSON file as a dictionary
with open("riotai/json_to_netcdf_maps/json_to_netcdf.json", "r") as file:
    json_netcdf_map = json.load(file)

# Filter the dictionary for entries with the key "3hr"
json_netcdf_3hr_map = json_netcdf_map.get("3hr", {})
json_netcdf_3hr_map = {k: v for k, v in json_netcdf_3hr_map.items() if k in keys}

### Inspect the JSON reference files for total references, dimension shape, and chunking strategy


In [6]:
# The variable to check.
VAR_KEY = "pr"

# Load the Kerchunk JSON file
for k in list(json_netcdf_3hr_map.keys())[:3]:
    with open(k, "r") as f:
        ref = json.load(f)

        zarray_key = f"{VAR_KEY}/.zarray"
        print(json.loads(ref["refs"][zarray_key]))
        zattrs_key = f"{VAR_KEY}/.zattrs"
        print(json.loads(ref["refs"][zattrs_key]))
        total_refs = len(ref["refs"])
        print("Total references:", total_refs)

{'shape': [189928, 512, 1024], 'chunks': [9, 30, 59], 'dtype': '<f4', 'fill_value': 1.0000000200408773e+20, 'order': 'C', 'filters': [{'id': 'shuffle', 'elementsize': 4}, {'id': 'zlib', 'level': 3}], 'dimension_separator': '.', 'compressor': None, 'zarr_format': 2}
{'_ARRAY_DIMENSIONS': ['time', 'lat', 'lon'], 'standard_name': 'precipitation_flux', 'long_name': 'Precipitation', 'units': 'kg m-2 s-1', 'cell_methods': 'area: time: mean', 'history': '2019-07-07T10:03:16Z altered by CMOR: Reordered dimensions, original order: j i time.', 'cell_measures': 'area: areacella', 'missing_value': 1.0000000200408773e+20}
Total references: 7027324
{'shape': [189927, 512, 1024], 'chunks': [9, 30, 59], 'dtype': '<f4', 'fill_value': 1.0000000200408773e+20, 'order': 'C', 'filters': [{'id': 'shuffle', 'elementsize': 4}, {'id': 'zlib', 'level': 3}], 'dimension_separator': '.', 'compressor': None, 'zarr_format': 2}
{'_ARRAY_DIMENSIONS': ['time', 'lat', 'lon'], 'standard_name': 'precipitation_flux', 'long_

**Observation**

- Kerchunk reference files contain **millions of references** (≈ 7 million in these cases).
- Each reference corresponds to an underlying HDF5 chunk, so fine-grained chunking over long time series leads to very large reference counts.
- Large reference counts significantly increase `open_dataset` time due to Python-level metadata parsing and object creation.

**Reference count behaviors**:

| Reference count | Expected behavior     |
| --------------- | --------------------- |
| < 100k          | Opens quickly         |
| 100k–500k       | Noticeable delay      |
| ~1M             | Very slow             |
| **> 5M**        | 🚨 Metadata-dominated |

**Key Takeaway**

When Kerchunk reference counts reach the millions, performance becomes **metadata-bound rather than I/O-bound**. In these cases, Kerchunk will be slower than `open_mfdataset`, and improving performance requires reducing reference counts (larger chunks, fewer variables, shorter time spans) or avoiding Kerchunk altogether for local, well-chunked netCDF data.


## Overall Takeaway

Kerchunk performance is dominated by **reference count**, not dimension order, for long, finely chunked 3hr datasets. When reference counts reach the millions, `open_dataset` becomes metadata-bound and slower than `open_mfdataset`. Kerchunk is best suited for reducing I/O costs in remote or cloud-native workflows; for local, well-chunked netCDF files with long time series, native netCDF access is often the better-performing option.
